# I'm Something of a Painter Myself — 동결 생성모델로 Monet 화풍 생성 (Colab)

**연습 기법** (IOAI 2024 *Madarian Cow* 와 동일): **사전학습 생성모델을 그대로(동결)** 써서 이미지를 만들고,
**인식모델이 그 결과를 평가**한다. (Madarian Cow: miniSD 생성 → DETR 검출 채점)

**시나리오 매핑**:
- Madarian Cow: 동결 디퓨전 생성 → **DETR** 인식으로 채점.
- 여기(Monet GAN): 동결 디퓨전(**sd-turbo img2img**)으로 사진을 **Monet 화풍**으로 변환 → 캐글이 **MiFID**
  (InceptionV3 특징 거리)로 채점. 둘 다 *생성 → 인식모델 평가* 패턴.

**이 대회는 Getting Started 라 상시 제출 가능**합니다. 코드는 `diffusers` 기본만(학습 없음 — 동결 모델만 사용).


## 0. 라이브러리 설치 & Kaggle 자격증명


In [ ]:
import sys, subprocess
for pkg in ["diffusers", "accelerate", "torch", "transformers", "kaggle", "pillow"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")


In [ ]:
import os

# Kaggle API 토큰 (내장)
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"

print("Kaggle 자격증명 설정 완료 (내장 토큰)")


## 1. Kaggle 에서 데이터 다운로드 (변환할 사진 photo_jpg)


In [ ]:
import os, glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.competition_download_files("gan-getting-started", path=WORK_DIR, quiet=False)
for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
    os.remove(zp)
photos = sorted(glob.glob(os.path.join(WORK_DIR, "photo_jpg", "*.jpg")))
print("변환할 사진 수:", len(photos))


## 2. 동결 생성모델 (sd-turbo img2img) — *학습하지 않음*
사전학습 디퓨전을 그대로 불러와, 사진을 Monet 인상파 화풍으로 변환(image-to-image).


In [ ]:
import torch
from diffusers import AutoPipelineForImage2Image
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = AutoPipelineForImage2Image.from_pretrained(
    "stabilityai/sd-turbo", torch_dtype=torch.float16 if device=="cuda" else torch.float32
).to(device)
pipe.set_progress_bar_config(disable=True)
PROMPT = "a Claude Monet impressionist oil painting, soft brushstrokes, pastel colors"
print("동결 생성모델 로드 완료 (sd-turbo img2img)")


## 3. 사진 → Monet 화풍 생성
제출은 7,000~10,000장이 필요하다. 전체 사진을 변환한다. (빠른 테스트는 `DEMO=True` 로 일부만)


In [ ]:
DEMO = False                      # True 면 64장만(빠른 확인). 실제 제출은 False.
targets = photos[:64] if DEMO else photos

out_dir = os.path.join(WORK_DIR, "images")
os.makedirs(out_dir, exist_ok=True)
for i, p in enumerate(targets):
    init = Image.open(p).convert("RGB").resize((256, 256))
    img = pipe(PROMPT, image=init, num_inference_steps=2, strength=0.55,
               guidance_scale=0.0).images[0].resize((256, 256))
    img.save(os.path.join(out_dir, f"{i:05d}.jpg"))
    if (i + 1) % 500 == 0:
        print(f"  {i+1}/{len(targets)} 변환")
print("생성 완료:", len(os.listdir(out_dir)), "장 (Monet 화풍, 256x256)")


## 4. 캐글 제출 파일 생성 (`images.zip`)


In [ ]:
submission_path = os.path.join(WORK_DIR, "images.zip")
with zipfile.ZipFile(submission_path, "w") as zf:
    for f in sorted(os.listdir(out_dir)):
        zf.write(os.path.join(out_dir, f), f)
print("images.zip 생성:", len(os.listdir(out_dir)), "장")


In [ ]:
try:
    from google.colab import files
    files.download(submission_path)
except Exception as e:
    print("자동 다운로드 불가:", e)
    print("submission.csv 위치:", submission_path)


## 🏁 제출하기
`images.zip` 을 [I'm Something of a Painter Myself](https://www.kaggle.com/competitions/gan-getting-started) 에 제출하세요 (Getting Started — 상시 제출, MiFID 채점).

**기법 요약**: 생성모델을 **학습하지 않고(동결)** 그대로 써서 이미지를 만들고, **인식모델(MiFID=InceptionV3)이 평가** — IOAI *Madarian Cow* 와 동일한 *생성→인식* 패턴.
점수(MiFID, 낮을수록 좋음)를 낮추려면: 프롬프트/`strength`/스텝수 조정, 더 좋은 디퓨전, 또는 인식모델로 생성물을 필터링(Madarian Cow 의 'magic' 처럼 생성 조건을 조정).

> 참고: 전체 ~7,000장 변환은 Colab GPU 로 수십 분 걸립니다. 빠른 확인은 위 셀의 `DEMO=True`(64장)로 형식만 확인하세요(단, 실제 채점에는 7,000장 이상 필요).
